# Estimator Examples

This notebook demonstrates the common workflow for ACM estimator classes. It will show you how to implement your own estimator, with custom backend if needed.

In [ ]:
# Global setup
import itertools
import logging
from pathlib import Path

import lsstypes
import matplotlib.pyplot as plt
import numpy as np
from estimators.helpers import (
    load_estimator_parameters,  # This requires mockfactory to work
    make_lagrangian_mock,
)

from acm.utils.logging import setup_logging

data_positions, boxsize = make_lagrangian_mock()
params = load_estimator_parameters()

logger = logging.getLogger("Estimator examples")
setup_logging()

## Backends

All Estimator classes depend on a backend, that can either be passed at initailization if it already exists or that can be loaded from a global registry. Backends own the particle fields, and meshes configurations.

This backend should expose 3 main methods that need to be implemented:
- `set_density_contrast`: sets the density contrast field
- `read_density_contrast`: reads the density contrast values at given positions
- `get_query_positions`: to query positions within the box - useful to read density contrast at those positions

There are also 4 properties that need to be implemented: `boxsize`, `boxcenter`, `cellsize`, `meshsize`

> If your estimator does not require a backend, it sould accept any backend to be compatible with different estimator calls (see below)


**All estimator and backends expect positions in 3D cartesian coordinates !** 

In [ ]:
from acm.estimators.galaxy_clustering.backends.base import (
    EstimatorBackend,
    register_backend,
)


# Let's create a backend
@register_backend("my_backend")
class MyBackend(EstimatorBackend):
    """My custom backend for the estimator."""

    def __init__(
        self,
        data_positions: np.ndarray,
        randoms_positions: np.ndarray | None = None,
        data_weights: np.ndarray | None = None,
        randoms_weights: np.ndarray | None = None, # At least those 4 parameters are required by the EstimatorBackend class.
        boxsize: float | None = None, # Extra parameters can be added to the constructor as needed for your backend.
        boxcenter: float | None = None,
        meshsize: int | None = None
    ) -> None:
        super().__init__(
            data_positions=data_positions,
            randoms_positions=randoms_positions,
            data_weights=data_weights,
            randoms_weights=randoms_weights,
        ) # EstimatorBackend performs checks on the arrays shapes to ensure they are consistent.

        # Initialize your backend here. In this example, we set the internal attributes.
        self._boxsize = boxsize
        self._boxcenter = boxcenter
        self._meshsize = meshsize

    @property
    def boxsize(self) -> np.ndarray:  # noqa: D102
        return np.full((3, ), self._boxsize)
    @property
    def boxcenter(self) -> np.ndarray:  # noqa: D102
        return np.full((3, ), self._boxcenter)
    @property
    def meshsize(self) -> np.ndarray:  # noqa: D102
        return np.full((3, ), self._meshsize)
    @property
    def cellsize(self) -> np.ndarray:  # noqa: D102
        return self.boxsize / self.meshsize

    def set_density_contrast(self, **kwargs) -> None:
        """Implement the logic to compute and register the density contrast."""
        logger.info(f"Density contrast set with parameters: {kwargs}")

    def read_density_contrast(self, positions: np.ndarray, resampler: str = "cic",) -> np.ndarray:
        """Implement the logic to read the density contrast."""
        logger.info(f"Reading density contrast at positions: {positions} with resampler: {resampler}")
        return np.zeros(len(positions))  # Return a dummy array of zeros

    def get_query_positions(self, method: str = "randoms", nquery: int | None = None, seed: int = 42) -> np.ndarray:
        """Implement the logic to get query positions."""
        logger.info(f"Getting query positions with method: {method}, nquery: {nquery}, seed: {seed}")
        rng = np.random.default_rng(seed)
        nquery = nquery if nquery is not None else 100  # Default to 100 if nquery is not provided
        return rng.random((nquery, 3)) * self.boxsize  # Return random positions within the boxsize

In [ ]:
my_backend = MyBackend(
    data_positions=data_positions,
    boxsize=boxsize,
    boxcenter=boxsize / 2,
    meshsize=64
)
my_backend.set_density_contrast()

# Then, we can access:
print("Box size:", my_backend.boxsize)
print("Box center:", my_backend.boxcenter)
print("Mesh size:", my_backend.meshsize)
print("Cell size:", my_backend.cellsize)

query = my_backend.get_query_positions(nquery=5)
print("Query positions:", query)
print("Density contrast at query positions:", my_backend.read_density_contrast(query))

### Notes:
1. The registration runs at the class definition. `acm` backends dependencies might not be installed, so the class needs to be imported at least once in the namespace for the registration to work.

> ```python
> load_backend('jaxpower') # will crash
> # ----------------
> from acm.estimators.galaxy_clustering.backends.jaxpower import JaxpowerBackend # registers the backend at import
> load_backend('jaxpower') # will not crash
> ```

2. Estimators can have extra public properties, methods and attributes, but those can only be accessed by estimators explicitely constraining the backend to a specific subclass of `EstimatorBackend` (e.g. the Power spectrum estimator). In the general case, do not expect to be able to access those extra properties.

3. The backends should not store the positions/weights arrays, as those should be stored in the estimator class - to avoid data duplication.

## Estimators

Estimators classes are designed to run with only 3 steps:
1. Initialize the class with a (common) backend
2. Compute the estimator
3. Save the output

Estimator classes expose 4 public methods:
- `compute`: to compute the estimator output as an `lsstype` object
- `save`: to save the output as a `.h5` file
- `load`: to load the saved file
- `plot`: a helper function to visualize the output

In [ ]:
from acm.estimators.galaxy_clustering.base import BaseEstimator
from acm.typing import LsstypeObject  # Either ObservableLeaf or ObservableTree


class MyEstimator(BaseEstimator):
    """My custom estimator."""

    def __init__(
        self,
        backend: str | EstimatorBackend,
        data_positions: np.ndarray,
        randoms_positions: np.ndarray | None = None,
        data_weights: np.ndarray | None = None,
        randoms_weights: np.ndarray | None = None,
        custom_value: float = 3,  # Any estimator-specific parameters needs to be defined explicitly.
        **kwargs, # Extra parameters can be added to the constructor as needed for your estimator.
    ) -> None:
        super().__init__(
            backend=backend,
            data_positions=data_positions,
            randoms_positions=randoms_positions,
            data_weights=data_weights,
            randoms_weights=randoms_weights,
            **kwargs,
        ) # Initializes the backend and stores positiosn and weights
        self._custom_value = custom_value  # Store the custom parameter for later use in the computation.

    def compute(self, **kwargs) -> LsstypeObject:
        """Implement the logic to compute the estimator."""
        logger.info(f"Computing estimator with parameters: {kwargs}")
        # Here you would implement the actual computation logic for your estimator.
        data = np.arange(self.backend.size_data) * self._custom_value
        return lsstypes.ObservableLeaf(
            data=data,
            index=np.arange(len(data)),
            coords=["index"],
            attrs={"name": "MyEstimatorResult"},
        )

    @staticmethod
    def load(filename: str | Path) -> LsstypeObject:
        """Load an estimator result from file."""
        # Here, you can add kwargs to the read function if you need extra parameters for loading the result.
        # e.g. projecting to multipoles, etc.
        return lsstypes.read(filename)

    @staticmethod
    def plot(
        obj: LsstypeObject,
        fig: plt.Figure | None = None,
        ax: plt.Axes | None = None,
    ) -> tuple[plt.Figure, plt.Axes]:
        """Plot the provided estimator result. Return figure and ax."""
        if fig is None or ax is None:
            fig, ax = plt.subplots()
            ax.set_xlabel("X-axis")
            ax.set_ylabel("Y-axis")
        # Here you would implement the actual plotting logic for your estimator result.
        ax.plot(obj.data, label=obj.attrs["name"])
        return fig, ax

When loaded, estimators can load a backend from registry, by passing the correct key. They can also load an existing backend by passing the instance at initialization

In [ ]:
my_estimator = MyEstimator(
    backend="my_backend", # passing the backend name will automatically use the registered backend class.
    data_positions=data_positions,
    boxsize=boxsize, # All extra parameters are passed to the backend constructor, so you can add any extra parameters you need for your backend.
    boxcenter=boxsize / 2,
    meshsize=64,
)

# If we need the density contrast, we need to set it by calling the backend's set_density_contrast method.
my_estimator.backend.set_density_contrast()

# then, we can compute the estimator:
result = my_estimator.compute()

my_estimator.plot(result)

In [ ]:
my_backend = MyBackend(
    data_positions=data_positions,
    boxsize=boxsize,
    boxcenter=boxsize / 2,
    meshsize=64
) # We define the backend instance beforehand, for example if we want to use the same backend for multiple estimators.

my_backend.set_density_contrast() # let's define the density contrast only once

my_estimator = MyEstimator(
    backend=my_backend, # passing the backend instance will use that instance directly.
    data_positions=data_positions,
    # We don't need to pass boxsize, boxcenter, meshsize again since we already defined the backend instance with those parameters
)

# then, we can compute the estimator:
result = my_estimator.compute()

my_estimator.plot(result)

## Estimator helpers

Here are some example of local helpers that can be used to handle several estimators in a script. As those are very sensitive to user choices, they are not included in acm, but are quite straightforward to reproduce.

In [ ]:
from acm.estimators.galaxy_clustering.base import BaseEstimator
from acm.estimators.galaxy_clustering.bispectrum import BispectrumMultipoles
from acm.estimators.galaxy_clustering.density_split import DensitySplit
from acm.estimators.galaxy_clustering.spectrum import PowerSpectrumMultipoles
from acm.estimators.galaxy_clustering.tpcf import TwoPointCorrelationFunctionEstimator


def get_estimator(name: str) -> type[BaseEstimator]:
    """Get the estimator class by alias name."""
    if name == "tpcf":
        return TwoPointCorrelationFunctionEstimator
    if name == "spectrum":
        return PowerSpectrumMultipoles
    if name == "bispectrum":
        return BispectrumMultipoles
    if name.startswith("ds_"): # ds_xiqq, ds_xiqg, ds_pkqq, ds_pkqg
        return DensitySplit
    raise ValueError(f"Unknown estimator name: {name}")

# load_estimator_parameters() reads from file a dictionary of default parameters for each estimator. 
# For each alias key, there is a "initialization" and "compute" dictionary of parameters, used at these respective steps.
def get_estimator_args(name: str) -> dict:
    """Get the default arguments for the estimator by alias name."""
    if name not in params:
        raise ValueError(f"No default parameters found for estimator: {name}")
    return params[name]

In [ ]:
from acm.estimators.galaxy_clustering.backends.jaxpower import (
    JaxpowerBackend,  # noqa: F401 - register the backend
)

stat_name = "spectrum"

cls = get_estimator(stat_name)
args = get_estimator_args(stat_name)
init_args = args.get("initialization", {})
compute_args = args.get("compute", {})
estimator = cls(backend="jaxpower", data_positions=data_positions, **init_args)
estimator.backend.set_density_contrast()  # Set the density contrast before computing the estimator
result = estimator.compute(los="z", **compute_args)  # ty:ignore[unknown-argument]

estimator.plot(result)

**To learn more about how other estimators work, check out the `nb/estimators/` folder, which contains all the example notebooks for the different estimators !**

# Compression Example

To compress several estiators output in the format required by the `Observable` class in the next step of the pipeline, we need to use the `Compressor` class. This class will read several `.h5` files, merge them, and compress them into a single `xarray.DataArray` object.

In [ ]:
# First, let's make a dummy lsstypes object:
s = np.linspace(0, 50, 51)
mu = np.linspace(-1, 1, 101)
rng = np.random.RandomState(seed=42)
labels = ['DD', 'DR', 'RR']
leaves = []
for _label in labels:
    counts = 1. + rng.uniform(size=(s.size, mu.size))
    leaves.append(lsstypes.ObservableLeaf(
        counts=counts,
        s=s,
        mu=mu,
        coords=['s', 'mu'],
        attrs=dict(los='x'),
    ))
tree = lsstypes.ObservableTree(leaves, pairs=labels)

# Now, let's create a function to generate the mock files in a temporary directory for testing the Compressor class.
def generate_mock_files(tmp_path: Path) -> None:
    """Generate mock .h5 files in a temporary directory for testing the Compressor class."""
    for i, j, k, v in itertools.product(range(2), range(2), range(1, 4), range(1, 2)):
        dir_path = tmp_path / f"I{i}/J{j}_K{k}/v{v}.0"
        dir_path.mkdir(parents=True, exist_ok=True)
        file_path = dir_path / "file_M.h5"
        lsstypes.write(tree, file_path)  # Write the mock lsstypes object to the .h5 file

generate_mock_files(tmp_path=Path("mock_data"))

First, we can identify the files we want to compress trough pattern matching. The `Compressor` class will create a `Pattern` object, that can confert a formatted string into a glob or regex pattern.

In [ ]:
from acm.estimators.compression import Pattern

root = Path("mock_data")
pattern_str = "I{i}/J{j}_K{k}/{l}/file_{m}.h5" # Let's replace the version by the l index to have a string value

pattern = Pattern(root=root, pattern=pattern_str)

print("Reconstructed glob pattern: ", pattern.to_glob())
print("Reconstructed regex pattern: ", pattern.to_regex())

In [ ]:
from acm.estimators.compression import Compressor

# First, the Compressor instance registers matching files trough glob pattern matching.
compressor = Compressor(root=root, pattern=pattern_str)

The compressor can then read the files in the list. We can specify indexes to ignore when reading the files. 
The data is stored in a `ObjectGroup` instance, which contains `IndexedObject` instances. The `ObjectGroup` can only contain `IndexedObject` instances with the same indexes.

In [ ]:
group = compressor.read(reader=lsstypes.read, ignore_index=["m"])

print("Names of the group indexes: ", group.names)

# We can select specific indices of the group !
subgroup = group.get(i=0, j=1, k=2)  # Selects the subgroup with i=0, j=1, k=2
print(f"Found {len(subgroup)} elements with i=0, j=1, k=2")

In the case where some indexes are ignored, there might be several files that have the same final indexes. In that case, you can use the `merge` method to merge the `IndexedObject` instances with the same indexes. The `merge` method takes a `method` argument, which must take a list of `lsstypes` objects and return a single `lsstypes` object.

In [ ]:
group = group.merge(method=lsstypes.mean)  # Merge the IndexedObject instances with the same indexes using the mean method

print(f"After merging, there are {len(group)} elements remaining in the group.")

If you need to apply transformations to the data, you can access the `lsstypes` object methods directly on the group (trough a `getattr` override). This will apply the method to all `lsstypes` objects in the group, and return a new `ObjectGroup` instance with the transformed data.

In [ ]:
group = group.select(s=(0, 50))  # Selects the s values between 0 and 50

The `ObjectGroup` class has some list properties. It is ordered in the order of the sorted files, which should usually be enough to keep the order of the indexes.

In the case where you want to change the order of the objects to match some nested index sorting, you can use the `sort` method, which will sort the `IndexedObject` instances in the group according to the specified order of indexes. The `sort` method takes a list of index names, and will sort the `IndexedObject` instances in the group according to the specified order of indexes.

> ***Note:** The `sort` method does not modify the index order of the `IndexedObject` instances, it only changes the order of the objects in the group. The index order is still determined by the `IndexedObject` instances themselves.*

In [ ]:
from acm.estimators.compression import IndexedObject, ObjectGroup

obj1 = IndexedObject(indexes={"i": 0, "j": 4}, data=tree)
obj2 = IndexedObject(indexes={"i": 1, "j": 3}, data=tree)
obj3 = IndexedObject(indexes={"i": 2, "j": 2}, data=tree)
example_group = ObjectGroup([obj1, obj2, obj3]) # By construction this object sorts by i, then j
print("Before sorting:", [obj.indexes for obj in example_group.objects])

sorted_group = example_group.sort("j", "i")  # Sorts the IndexedObject instances in the group according to the order of indexes j, then i
print("After sorting:", [obj.indexes for obj in sorted_group.objects])

In the case of the compression, the sorting is handled internally. If the sorting order matches the entire index name list, the dimensions will be ordered accordingly (see the `compress` method).
Otherwise, it is not possible to infer the expected order, so the dimensions will not be reordered.

Once merged and transformed, the `ObjectGroup` can be compressed into a single `xarray.DataArray` object. The `compress` method takes an optional `order` argument for sorting, and an optional `reindex` argument for reindexing some nested indexes (in the case of sparse nested indexes, to avoid NaN values in the final `xarray.DataArray` object). The `reindex` argument takes a dictionary of each index to reindex, and the list of values to reindex to. The `compress` method will return a single `xarray.DataArray` object with the compressed data.

In [ ]:
result = Compressor.compress(
    data=group,
    order=["i", "j", "k"],  # Optional: specify the order of indexes for sorting.
    reindex={"k": ["i", "j"]},  # Optional: specify the reindexing for some nested indexes to avoid NaN values in the final xarray.DataArray object.
    drop_single = True  # Optional: drop single-dimensional coordinates in the final xarray.DataArray object.
)

result

**To learn more about this format, check out the Observable documentation.**